
# Project Management Analytics

This project demonstrates an end‑to‑end analytics workflow using a **synthetic project management dataset**. The objective is to explore factors that influence project outcomes and build predictive models to estimate the probability of project success.

The dataset contains 1,000 synthetic project records with the following columns:

- `project_id`: Unique identifier for each project.
- `team_size`: Number of people on the project team.
- `team_experience_years`: Average years of professional experience of team members.
- `budget_thousands`: Budget allocated to the project (in thousands of dollars).
- `duration_days`: Estimated duration of the project in days.
- `complexity`: Categorized complexity level (`Low`, `Medium`, `High`).
- `risk`: Risk score between 0 and 1 (higher values indicate greater risk).
- `success`: Binary outcome indicating whether the project was successful (1) or not (0).

We'll perform exploratory data analysis (EDA) to understand the relationships among variables and then build predictive models (logistic regression and random forest) to predict project success. You'll see how model performance can improve by adding complexity.


In [ ]:

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Set visualisation style
sns.set(style="whitegrid")

# Load dataset
file_path = 'synthetic_project_data.csv'
data = pd.read_csv(file_path)

# Display first few rows
data.head()


In [ ]:

# Describe numeric columns
data.describe()

# Check for missing values
data.isnull().sum()


In [ ]:

# Distribution of numeric features
numeric_cols = ['team_size', 'team_experience_years', 'budget_thousands', 'duration_days', 'risk']
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(15, 8))
for i, col in enumerate(numeric_cols):
    ax = axes[i//3, i%3]
    sns.histplot(data[col], kde=True, ax=ax)
    ax.set_title(f'Distribution of {col}')

plt.tight_layout()
plt.show()

# Countplot for complexity and success
plt.figure(figsize=(6,4))
sns.countplot(x='complexity', hue='success', data=data)
plt.title('Project Success by Complexity Level')
plt.show()

# Correlation heatmap for numeric features
plt.figure(figsize=(8,6))
corr = data[['team_size', 'team_experience_years', 'budget_thousands', 'duration_days', 'risk', 'success']].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix')
plt.show()


In [ ]:

# Separate features and target
y = data['success']
X = data.drop(columns=['success', 'project_id'])

# Identify categorical and numeric columns
categorical_cols = ['complexity']
numeric_cols = ['team_size', 'team_experience_years', 'budget_thousands', 'duration_days', 'risk']

# Preprocess data: one‑hot encode categorical variables
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first'), categorical_cols),
    ],
    remainder='passthrough'
)

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Fit logistic regression model
log_reg = LogisticRegression(max_iter=1000)

from sklearn.pipeline import Pipeline

log_reg_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                         ('model', log_reg)])

log_reg_pipeline.fit(X_train, y_train)

# Predict and evaluate
preds_lr = log_reg_pipeline.predict(X_test)
print("Logistic Regression Classification Report:")
print(classification_report(y_test, preds_lr))

# Confusion matrix
cm_lr = confusion_matrix(y_test, preds_lr)
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - Logistic Regression')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# Fit RandomForest model
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                   ('model', rf)])
rf_pipeline.fit(X_train, y_train)

# Predict and evaluate
preds_rf = rf_pipeline.predict(X_test)
print("Random Forest Classification Report:")
print(classification_report(y_test, preds_rf))

# Confusion matrix
cm_rf = confusion_matrix(y_test, preds_rf)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Greens')
plt.title('Confusion Matrix - Random Forest')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# Feature importance from random forest (after one-hot encoding)
# Get feature names after preprocessing
onehot_feature_names = log_reg_pipeline.named_steps['preprocessor'].named_transformers_['cat'].get_feature_names_out(categorical_cols)

# Combine with numeric columns
feature_names = list(onehot_feature_names) + numeric_cols

importances = rf_pipeline.named_steps['model'].feature_importances_

# Create a DataFrame for visualization
feat_imp_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
feat_imp_df = feat_imp_df.sort_values('importance', ascending=False)

plt.figure(figsize=(8,5))
sns.barplot(x='importance', y='feature', data=feat_imp_df)
plt.title('Feature Importances - Random Forest')
plt.show()



## Conclusions

The synthetic project management dataset allows us to explore how various factors correlate with project success. From the exploratory analysis and models, we observed:

- **Team experience and budget** positively correlate with project success, while **higher risk** and **higher complexity** lower success rates.
- A logistic regression provides a baseline model for predicting success. It captures linear relationships but may not handle complex interactions well.
- A random forest, which can model non‑linear relationships and interactions, produced better classification performance compared with logistic regression.

These results illustrate how increasingly complex models can improve predictive performance for project outcome analytics. This repository is an example of how to structure a data analysis project with clear documentation, EDA, and modeling steps to demonstrate analytical skills for roles such as business analyst, program manager, or data analyst.
